# LangChain L3 — Level 2 — Your first `create_agent()`
OpsPilot v2 gets several tools and LangChain's standard agent constructor. `create_agent()`
packages the loop from L2 as a compiled **LangGraph** graph with two nodes, *model* and
*tools*, and a conditional edge between them.

```text
              +---------+   tool calls?   +---------+
  START --->  |  model  | -------yes----> |  tools  |
              +---------+                 +---------+
                   | no                        |
                   v                           |
                  END   <----------------------+  (back to model)
```

A **chain** has a fixed order of steps, A -> B -> C. An **agent** chooses the order at run
time: A -> C -> C -> B, depending on what it discovers. That is the only real difference, and
it is why agents need limits, logging and approvals that chains do not.

### Step 1 — OpsPilot's first real tools

Small fake data stands in for the company's CRM, order system and a weather API. The tools are
deliberately simple; section L4 hardens them.

In [ ]:
# ours: Meridian's tiny fake data (see "Meet OpsPilot" at the top)
CUSTOMERS = {
    "C001": {"name": "Alice Fernandes", "plan": "Pro",        "email": "alice@example.com", "since": "2024-03-01"},
    "C002": {"name": "Bob Iyer",        "plan": "Enterprise", "email": "bob@example.com",   "since": "2022-11-15"},
    "C003": {"name": "Chen Wei",        "plan": "Starter",    "email": "chen@example.com",  "since": "2026-07-20"},
}
ORDERS = {
    "O1001": {"customer_id": "C001", "item": "Router X200",   "amount": 120.0,  "status": "delivered", "days_ago": 12},
    "O1002": {"customer_id": "C002", "item": "Server rack",   "amount": 500.0,  "status": "charged twice", "days_ago": 3},
    "O1003": {"customer_id": "C003", "item": "Cable bundle",  "amount": 45.0,   "status": "shipped",   "days_ago": 45},
}
WEATHER = {"Mumbai": "32°C and humid", "London": "14°C and rainy", "New York": "18°C and cloudy"}

@tool                                    # LangChain decorator; each function body is ours
def get_weather(city: str) -> str:
    """Get the current weather for a city (used for delivery planning)."""
    return WEATHER.get(city, f"weather unavailable for {city}")

@tool
def get_customer(customer_id: str) -> str:
    """Retrieve a customer record from the CRM by customer id, e.g. 'C001'."""
    record = CUSTOMERS.get(customer_id)
    return json.dumps({"id": customer_id, **record} if record else {"error": "customer_not_found", "customer_id": customer_id})

@tool
def get_order(order_id: str) -> str:
    """Retrieve an order from the order system by order id, e.g. 'O1001'."""
    return json.dumps(ORDERS.get(order_id, {"error": "order_not_found", "order_id": order_id}))

OPSPILOT_TOOLS_V2 = [calculate, get_weather, get_customer, get_order]
print("tools:", [t.name for t in OPSPILOT_TOOLS_V2])

### Step 2 — Create the agent and run it

`create_agent()` takes a model (an instance here, because OpenRouter needs a custom base URL;
`"openai:gpt-5.4"` style strings also work for direct providers), a list of tools and a system
prompt. The input and output are a **state dictionary** whose `messages` key holds the whole
trajectory. The last message is the answer; the messages before it are the evidence.

In [ ]:
from langchain.agents import create_agent           # LangChain: the standard agent constructor

OPSPILOT_PROMPT = """You are OpsPilot, an operations assistant for Meridian Supply Co.
You have tools for arithmetic, weather, customer lookup and order lookup.
Use tools whenever they give more reliable information than your own knowledge.
Answer concisely and mention the ids you looked up."""

opspilot = create_agent(model=model, tools=OPSPILOT_TOOLS_V2, system_prompt=OPSPILOT_PROMPT)   # LangChain -> returns a LangGraph graph

result = opspilot.invoke({"messages": [{"role": "user", "content": "My customer is C001. What plan are they on, and what's the weather in Mumbai?"}]})   # LangGraph: run the graph on a state dict

print("ANSWER:", text_of(result["messages"][-1]), "\n")           # result["messages"] is the LangGraph state
print("TRAJECTORY:")
show_messages(result["messages"])                                  # ours

### Step 3 — Look under the hood

The agent is a compiled LangGraph **graph**. Three words to keep in mind from now on:

- **State** is a dictionary that flows through the graph. For an agent it holds `messages`.
- **Nodes** are functions that read the state and return an update. The agent has two: *model* and *tools*.
- **Edges** connect nodes. A *conditional edge* picks the next node from the state
  ("tool calls present? go to *tools*, otherwise finish").

Printing the nodes shows exactly the two-node loop from the diagram above. L12 builds such graphs
by hand, starting with toy examples that have no model in them at all.

In [ ]:
print("graph nodes :", [name for name in opspilot.get_graph().nodes if not name.startswith("__")])   # LangGraph: inspect the compiled graph

# A question that needs two DEPENDENT tool calls: the order must be read before the customer id is known,
# so the loop runs model -> tools -> model -> tools -> model. Compare with the parallel calls above.
result = opspilot.invoke({"messages": [{"role": "user", "content": "Who placed order O1002 and what is their plan? Look up the customer too."}]})
show_messages(result["messages"])

### Recap

- **Problem seen:** hand-written loops grow features (limits, memory, approvals) that every project rewrites.
- **Layer added:** `create_agent()`: the same loop as a compiled LangGraph graph with a standard state shape.
- **Evidence:** the trajectory shows the model choosing tools and their order at run time.